In [5]:
import sys
sys.path.append('..')
sys.path.append('../../')
sys.path.append('../../..')

import copy
import os
import re
import json

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from tidepool_data_science_simulator.models.sensor import NoisySensor, IdealSensor
from tidepool_data_science_simulator.models.patient import VirtualPatientModel
from tidepool_data_science_simulator.models.pump import ContinuousInsulinPump
from tidepool_data_science_simulator.models.controller import LoopController, DoNothingController
from tidepool_data_science_simulator.evaluation.inspect_results import load_results
from tidepool_data_science_simulator.makedata.make_simulation import get_canonical_simulation
from tidepool_data_science_simulator.makedata.make_patient import (
    get_canonical_risk_pump_config,
    get_canonical_virtual_patient_model_config,
    get_canonical_sensor_config,
    DATETIME_DEFAULT
)

from tidepool_data_science_simulator.visualization.sim_viz import plot_sim_results
from tidepool_data_science_simulator.run import run_simulations

from tidepool_data_science_simulator.models.events import CarbTimeline, BolusTimeline
from tidepool_data_science_simulator.models.measures import Carb, Bolus

import datetime

import matplotlib.dates as mdates
formatter = mdates.DateFormatter('%H:%M')
cmap = plt.cm.plasma_r

In [6]:
def build_metabolic_sensitivity_sims(start_glucose_value=110, basal_rate=0.3, cir=20.0, isf=150.0, target_range_min=100, target_range_max=120, carb_timeline=None, bolus_timeline=None, duration_hrs=1, controller=LoopController):
    """
    Look at resulting bgs from settings that are correct/incorrect for analysis.

    Parameters
    ----------
    scenario_csv_filepath: str
        Path to the scenario file

    param_grid: list of dicts
        Parameters to vary
    """
    
    
    
    sims = {}
    for exercise_preset_p in np.arange(0.1, 1.09, 0.1):
        t0, patient_config = get_canonical_virtual_patient_model_config(start_glucose_value = start_glucose_value, basal_rate=basal_rate, cir=cir, isf=isf, carb_timeline=carb_timeline, bolus_timeline=bolus_timeline) # patient has many attributes e.g. starting glucose (default: 110), recommendatio accept probability, etc.    
        
        t0, sensor_config = get_canonical_sensor_config(t0, start_value = start_glucose_value) # sensor config has a blood glucose history. right now looks like the starting value repeated 'n' times every 5 minutes before t0
        

        t0, pump_config = get_canonical_risk_pump_config(t0, basal_rate=basal_rate, cir=cir, isf=isf, target_range_min=target_range_min, target_range_max=target_range_max, carb_timeline=carb_timeline, bolus_timeline=bolus_timeline) # sets a carb timeline, bolus timeline (both initialized with 0 at t=0, assuming new values are then added on), basal schedule (e.g. 0.3 units delivered for 24 hours), carb ratio schedule (e.g. constant carb ratio of 20 for 24 hours). similarly for ISR and target range. ques: is there a schedule different from 24 hours?

        patient_config.recommendation_accept_prob = 1.0  # Accept the bolus
        
        
        basal_p_factor = -1 + exercise_preset_p  # note: funky math because of how override function works
        pump_config.basal_schedule.set_override(basal_p_factor)

        isf_p_factor = -1 + 1 / (exercise_preset_p)  # note: funky math because of how override function works
        pump_config.insulin_sensitivity_schedule.set_override(isf_p_factor)

        cir_p_factor = -1 + 1 / (exercise_preset_p)  # note: funky math because of how override function works
        pump_config.carb_ratio_schedule.set_override(cir_p_factor)
        
        # update patient config with preset too
        # patient_config.basal_schedule.set_override(basal_p_factor)
        # patient_config.insulin_sensitivity_schedule.set_override(isf_p_factor)
        # patient_config.carb_ratio_schedule.set_override(cir_p_factor)

        sensor_config.std_dev = 1.0 # ques: what does this do

        t0, sim = get_canonical_simulation(
            patient_config=patient_config,
            patient_class=VirtualPatientModel,
            sensor_config=sensor_config,
            # sensor_class=NoisySensor,
            sensor_class=IdealSensor,
            pump_config=pump_config,
            pump_class=ContinuousInsulinPump,
            controller_class=controller,
            multiprocess=True,
            duration_hrs=duration_hrs,
        )

        sim_id = f"Preset_{exercise_preset_p:.1f}"
        sims[sim_id] = sim

        pump_config.basal_schedule.unset_override()
        pump_config.insulin_sensitivity_schedule.unset_override()
        pump_config.carb_ratio_schedule.unset_override()
        
        # unset preset override in patient config
        # patient_config.basal_schedule.unset_override()
        # patient_config.insulin_sensitivity_schedule.unset_override()
        # patient_config.carb_ratio_schedule.unset_override()

    return sims

In [7]:
# simulation

save_date = '0514'

start_glucose_values = range(70, 270, 20)
basal_rate = 0.8
cir = 8.0
isf = 50.0    

target_range_min = 100
target_range_max = 120

t0 = DATETIME_DEFAULT
carb_timeline = None
bolus_timeline = None

# controller = LoopController
controller = DoNothingController

duration_hrs_list = [1, 2, 4, 8] # [1, ..., 8]

results = []

for start_glucose_value in start_glucose_values:
    for duration_hrs in duration_hrs_list:
        # this is only for convenience, but not for running speed
        # to get the number in the quickest way, just simulate for 8 hours

        sims = build_metabolic_sensitivity_sims(
            start_glucose_value=start_glucose_value,
            basal_rate=basal_rate,
            cir=cir,
            isf=isf,
            target_range_min=target_range_min,
            target_range_max=target_range_max,
            carb_timeline=carb_timeline,
            bolus_timeline=bolus_timeline,
            duration_hrs=duration_hrs,
            controller=controller
        )

        all_results, summary_results_df = run_simulations(sims,
                                                        save_dir=None,
                                                        save_results=False,
                                                        num_procs=100)

        fig1, ax1 = plt.subplots(dpi=150)
        fig2, ax2 = plt.subplots(dpi=150)

        for sim_id, results_df in all_results.items():
            preset = float(sim_id[-3:])
            bolus_insulin_total = results_df["true_bolus"].sum()
            basal_insulin_total = results_df["delivered_basal_insulin"].sum()

            entry = {
                'preset': preset,
                'bolus': bolus_insulin_total,
                'basal': basal_insulin_total,
                'start_bg': start_glucose_value,
                'hours': duration_hrs
            }

            results.append(entry)

            # get the simulated results
            plot_df = results_df.reset_index().iloc[-12 * duration_hrs:]

            ax1.plot(plot_df.time, plot_df.bg, label=sim_id, c=cmap(preset))

            ax2.plot(plot_df.time, plot_df.delivered_basal_insulin, label=sim_id, c=cmap(preset))

        ax1.legend()
        ax1.xaxis.set_major_formatter(formatter)
        ax1.set_xlabel('Time')
        ax1.set_ylabel('Blood Glucose (mg/dL)')
        ax1.set_title(f'Starting BG {start_glucose_value}\nDuration {duration_hrs}h')

        plt.tight_layout()
        fig1.savefig(f'./figures/{save_date}-bg-bg{start_glucose_value}-hrs{duration_hrs}.png')

        plt.close(fig1)

        ax2.legend()
        ax2.xaxis.set_major_formatter(formatter)
        ax2.set_xlabel('Time')
        ax2.set_ylabel('Delivered Basal Insulin (U)')
        ax2.set_title(f'Starting BG {start_glucose_value}\nDuration {duration_hrs}h')

        plt.tight_layout()
        fig2.savefig(f'./figures/{save_date}-basal-bg{start_glucose_value}-hrs{duration_hrs}.png')

        plt.close(fig2)


# convert result into a dataframe
df = pd.DataFrame(results)

tidepool_data_science_simulator.run: DEBUG    Results Directory: None
tidepool_data_science_simulator.run: DEBUG    Current Code Commit: ace416f
tidepool_data_science_simulator.run: DEBUG    Running: Preset_0.1. 1 of 10
tidepool_data_science_simulator.run: DEBUG    Running: Preset_0.2. 2 of 10
tidepool_data_science_simulator.run: DEBUG    Running: Preset_0.3. 3 of 10
tidepool_data_science_simulator.run: DEBUG    Running: Preset_0.4. 4 of 10
tidepool_data_science_simulator.run: DEBUG    Running: Preset_0.5. 5 of 10
tidepool_data_science_simulator.run: DEBUG    Running: Preset_0.6. 6 of 10
tidepool_data_science_simulator.run: DEBUG    Running: Preset_0.7. 7 of 10
tidepool_data_science_simulator.run: DEBUG    Running: Preset_0.8. 8 of 10
tidepool_data_science_simulator.run: DEBUG    Running: Preset_0.9. 9 of 10
tidepool_data_science_simulator.run: DEBUG    Running: Preset_1.0. 10 of 10
tidepool_data_science_simulator.run: DEBUG    Batch run time: 0.00m
tidepool_data_science_simulator.run:

In [8]:
results_df[results_df.active == 1].delivered_basal_insulin

time
2019-08-15 12:00:00    0.066667
2019-08-15 12:05:00    0.066667
2019-08-15 12:10:00    0.066667
2019-08-15 12:15:00    0.066667
2019-08-15 12:20:00    0.066667
                         ...   
2019-08-15 19:40:00    0.066667
2019-08-15 19:45:00    0.066667
2019-08-15 19:50:00    0.066667
2019-08-15 19:55:00    0.066667
2019-08-15 20:00:00    0.066667
Name: delivered_basal_insulin, Length: 97, dtype: float64

In [9]:
for duration_hrs in duration_hrs_list:

    bg_preset_matrix = df[df.hours == duration_hrs].pivot_table(index='start_bg', columns='preset', values='basal')

    fig, ax = plt.subplots(figsize=(6, 6), dpi=150)

    sns.heatmap(bg_preset_matrix, cmap="GnBu", annot=True, fmt="0.2f")

    plt.xlabel('Preset')
    plt.ylabel('Starting Blood Glucose (mg/dL)')
    plt.title(f'Duration {duration_hrs}h')
    plt.tight_layout()
    plt.savefig(f'./figures/{save_date}-fixed-duration{duration_hrs}-heatmap.png')
    plt.close()